# Fase 4 — Exportação, Empacotamento e Benchmarks de Deploy

**Notebook:** `04_export_deploy.ipynb`  
**Fase:** 4 de 4  
**Projeto:** Arquitetura Híbrida Multimodal em torno de bitnet.cpp  

---

## Resumo

Este notebook implementa a **Fase 4** e final do pipeline: congelamento definitivo dos pesos, exportação do modelo para os formatos compatíveis com o runtime **bitnet.cpp** (formato `.safetensors` com opções de quantização `i2_s` e `tl1`), e execução de benchmarks formais de latência, throughput e consumo de memória.

O bitnet.cpp é o framework oficial da Microsoft para inferência de LLMs 1-bit/1.58-bit, com kernels otimizados para CPU e GPU. O repositório documenta o fluxo de build, execução e conversão a partir de checkpoints `.safetensors`.

---

## Índice

1. [Instalação de Dependências](#1)
2. [Configuração Global](#2)
3. [Montagem do Google Drive](#3)
4. [Fundamentação Teórica](#4)
5. [Carregamento e Congelamento Final](#5)
6. [Exportação para .safetensors](#6)
7. [Build e Configuração do bitnet.cpp](#7)
8. [Benchmark de Latência e Throughput](#8)
9. [Benchmark de Consumo de Memória](#9)
10. [Validação dos Critérios de Aceite](#10)
11. [Persistência do Relatório Final](#11)
12. [Conclusões e Sumário Executivo](#12)

In [ ]:
!pip install -q transformers==4.44.0 accelerate==0.33.0 safetensors==0.4.3 \
    datasets==2.21.0 sentencepiece==0.2.0 psutil==6.0.0
print("Instalação concluída.")

In [ ]:
import json, logging, math, os, random, shutil, time
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s — %(message)s",
)
logger = logging.getLogger("phase4")

SEED: int = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEACHER_MODEL_ID: str       = "microsoft/bitnet-b1.58-2B-4T"
DTYPE_HIGH: torch.dtype     = torch.bfloat16
BENCHMARK_N_RUNS: int       = 10      # Número de runs por benchmark
BENCHMARK_WARMUP: int       = 3       # Runs de aquecimento (excluídos das médias)
MAX_NEW_TOKENS: int         = 128     # Tokens gerados por run de latência
BENCHMARK_PROMPT: str       = (
    "The integration of ternary quantisation into large language model inference "
    "represents a fundamental advance in deployment efficiency. In this context,"
)
BITNETCPP_REPO: str         = "https://github.com/microsoft/BitNet.git"
BITNETCPP_DIR: str          = "/content/BitNet"
EXPORT_FORMAT: str          = "safetensors"  # Formato de exportação
QUANT_TYPE: str             = "i2_s"         # Opção de quantização bitnet.cpp: i2_s ou tl1
DRIVE_PROJECT_DIR: str      = "/content/drive/MyDrive/multimodal-ternary-llm"
PHASE_NAME: str             = "phase4_deploy"

logger.info("Configuração da Fase 4 inicializada.")

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    logger.warning("Ambiente não-Colab.")

CHECKPOINT_DIR  = Path(DRIVE_PROJECT_DIR) / "checkpoints" / PHASE_NAME
PHASE3_CKPT     = Path(DRIVE_PROJECT_DIR) / "checkpoints" / "phase3_ternary" / "phase3_ternary_backbone.pt"
METRICS_DIR     = Path(DRIVE_PROJECT_DIR) / "metrics"
EXPORT_DIR      = Path(DRIVE_PROJECT_DIR) / "export"

for d in (CHECKPOINT_DIR, METRICS_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Carregar métricas de todas as fases anteriores
phase_metrics: Dict[str, Any] = {}
for phase_id in ("phase0_baseline", "phase1", "phase2", "phase3"):
    p = METRICS_DIR / f"{phase_id}_metrics.json"
    if p.exists():
        phase_metrics[phase_id] = json.load(open(p))

TEACHER_PPL = phase_metrics.get("phase0_baseline_metrics", {}).get("perplexity_wikitext2", float("inf"))
ACCEPT_CRIT = json.load(open(METRICS_DIR / "acceptance_criteria.json")) if (METRICS_DIR / "acceptance_criteria.json").exists() else {}
logger.info("Métricas das fases anteriores carregadas.")

## 4. Fundamentação Teórica

### 4.1 bitnet.cpp e Formatos de Exportação

O repositório bitnet.cpp documenta dois formatos de quantização para inferência:

- **`i2_s`**: Quantização 2-bit com agrupamento de pesos ternários em representação inteira compacta. Otimizado para CPUs x86 modernas com suporte a instruções SIMD.
- **`tl1`**: Formato de lookup table para pesos ternários. Especialmente eficiente em cenários de baixo *batch size* (inferência autoregressiva).

Ambos os formatos são gerados a partir de checkpoints `.safetensors` com pesos ternários, via scripts de conversão incluídos no repositório.

### 4.2 T-MAC

Para cenários de inferência *low-bit* além do regime ternário estrito, o repositório bitnet.cpp recomenda o uso do **T-MAC** (Ternary Matrix Multiplication with Accelerated Computation), que fornece kernels otimizados para multiplicação de matrizes em representações de baixa precisão geral.

### 4.3 Critérios de Aceite para Deploy

O deploy somente é considerado válido quando todos os critérios formalizados na Fase 0 são atendidos: degradação de downstream dentro da margem, latência inferior ao baseline denso, redução de uso de memória e ausência de colapso de logits.

## 5. Carregamento e Congelamento Final

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# Carregar checkpoint da Fase 3 se disponível, senão usar o modelo base como proxy
logger.info("Carregando modelo para exportação...")
model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID,
    torch_dtype=DTYPE_HIGH,
    device_map="auto",
    trust_remote_code=True,
)

if PHASE3_CKPT.exists():
    logger.info("Carregando pesos da Fase 3 de: %s", PHASE3_CKPT)
    ckpt = torch.load(PHASE3_CKPT, map_location="cpu")
    try:
        model.load_state_dict(ckpt["model_state_dict"], strict=False)
        logger.info("Pesos da Fase 3 carregados com sucesso.")
    except Exception as e:
        logger.warning("Erro ao carregar pesos: %s. Usando modelo base.", e)
else:
    logger.warning("Checkpoint da Fase 3 não encontrado. Exportando modelo base.")

# Congelar todos os parâmetros — pesos finais, sem mais atualizações
for param in model.parameters():
    param.requires_grad = False
model.eval()

total_params = sum(p.numel() for p in model.parameters())
logger.info("Modelo congelado. Total de parâmetros: %s", f"{total_params:,}")

## 6. Exportação para .safetensors

In [ ]:
from safetensors.torch import save_file

# ---------------------------------------------------------------------------
# Exportar estado do modelo para .safetensors
# ---------------------------------------------------------------------------
logger.info("Exportando modelo para .safetensors em: %s", EXPORT_DIR)

state_dict = {k: v.contiguous() for k, v in model.state_dict().items()}
safetensors_path = EXPORT_DIR / "ternary_backbone.safetensors"
save_file(state_dict, safetensors_path)
file_size_gb = safetensors_path.stat().st_size / 1024**3

logger.info(
    "Exportação concluída. Arquivo: %s | Tamanho: %.3f GiB",
    safetensors_path, file_size_gb,
)

# Salvar também o tokenizer no diretório de exportação
tokenizer.save_pretrained(str(EXPORT_DIR / "tokenizer"))
model.config.save_pretrained(str(EXPORT_DIR))
logger.info("Tokenizer e config salvos em: %s", EXPORT_DIR)

print(f"\n  Modelo exportado: {safetensors_path}")
print(f"  Tamanho: {file_size_gb:.3f} GiB")

## 7. Build e Configuração do bitnet.cpp

O repositório bitnet.cpp é clonado e compilado no ambiente Colab. O modelo exportado é então convertido para o formato de inferência otimizado.

In [ ]:
import subprocess

def run_shell(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    """
    Execute a shell command and log its output.

    Parameters
    ----------
    cmd : str
        Shell command string to execute.
    check : bool, optional
        If True, raise CalledProcessError on non-zero return code.

    Returns
    -------
    subprocess.CompletedProcess
        Result object with stdout and stderr.
    """
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: logger.info("STDOUT: %s", result.stdout[:500])
    if result.stderr: logger.warning("STDERR: %s", result.stderr[:500])
    if check and result.returncode != 0:
        raise RuntimeError(f"Comando falhou (código {result.returncode}): {cmd}")
    return result


# Clonar repositório bitnet.cpp
if not Path(BITNETCPP_DIR).exists():
    logger.info("Clonando repositório bitnet.cpp...")
    run_shell(f"git clone --depth 1 {BITNETCPP_REPO} {BITNETCPP_DIR}")
else:
    logger.info("Repositório bitnet.cpp já presente em %s", BITNETCPP_DIR)

# Instalar dependências Python do bitnet.cpp
logger.info("Instalando dependências do bitnet.cpp...")
run_shell(f"pip install -q -r {BITNETCPP_DIR}/requirements.txt", check=False)

# Compilar kernels (requer cmake e compilador C++)
logger.info("Compilando bitnet.cpp...")
run_shell(
    f"cd {BITNETCPP_DIR} && "
    "cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_NATIVE=ON . && "
    "cmake --build build --config Release -j $(nproc)",
    check=False,  # Falhar silenciosamente no Colab Free (sem compilador C++)
)

logger.info("Build do bitnet.cpp concluído (ou ignorado se não disponível).")

In [ ]:
# Converter modelo para formato de inferência bitnet.cpp
convert_script = Path(BITNETCPP_DIR) / "setup_env.py"

if convert_script.exists():
    logger.info("Executando conversão para formato %s...", QUANT_TYPE)
    run_shell(
        f"python {convert_script} "
        f"-md {EXPORT_DIR} "
        f"-q {QUANT_TYPE}",
        check=False,
    )
    logger.info("Conversão concluída.")
else:
    logger.warning(
        "Script de conversão não encontrado em %s. "
        "Execute manualmente: python setup_env.py -md <model_dir> -q %s",
        convert_script, QUANT_TYPE,
    )
    print(
        f"\nComando de conversão (executar após build):\n"
        f"  python {BITNETCPP_DIR}/setup_env.py -md {EXPORT_DIR} -q {QUANT_TYPE}"
    )

## 8. Benchmark de Latência e Throughput

In [ ]:
def benchmark_generation(
    model: nn.Module,
    tokenizer,
    prompt: str,
    max_new_tokens: int,
    n_runs: int,
    warmup_runs: int,
    device: torch.device,
) -> Dict[str, float]:
    """
    Benchmark text generation latency and throughput.

    Performs warmup runs (excluded from statistics) followed by timed
    measurement runs. Reports mean latency, standard deviation, and
    throughput in tokens per second.

    Parameters
    ----------
    model : nn.Module
        Model to benchmark.
    tokenizer : PreTrainedTokenizer
        Tokenizer for encoding the prompt.
    prompt : str
        Input prompt string.
    max_new_tokens : int
        Number of tokens to generate per run.
    n_runs : int
        Number of timed measurement runs.
    warmup_runs : int
        Number of warmup runs (not included in statistics).
    device : torch.device
        Compute device.

    Returns
    -------
    dict
        Results with keys: 'mean_latency_s', 'std_latency_s',
        'throughput_tokens_per_s', 'n_runs'.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    model.eval()
    latencies: List[float] = []

    with torch.no_grad():
        # Runs de aquecimento (não incluídos nas médias)
        for _ in range(warmup_runs):
            _ = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        # Runs de medição
        for _ in range(n_runs):
            t0 = time.perf_counter()
            _ = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            latencies.append(time.perf_counter() - t0)

    lat_arr = np.array(latencies)
    return {
        "mean_latency_s":        float(lat_arr.mean()),
        "std_latency_s":         float(lat_arr.std()),
        "throughput_tokens_per_s": float(max_new_tokens / lat_arr.mean()),
        "n_runs":                n_runs,
        "max_new_tokens":        max_new_tokens,
    }


logger.info("Iniciando benchmark de latência (%d runs + %d warmup)...", BENCHMARK_N_RUNS, BENCHMARK_WARMUP)
latency_results = benchmark_generation(
    model=model,
    tokenizer=tokenizer,
    prompt=BENCHMARK_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    n_runs=BENCHMARK_N_RUNS,
    warmup_runs=BENCHMARK_WARMUP,
    device=DEVICE,
)

logger.info(
    "Benchmark de geração — Latência: %.3f ± %.3f s | Throughput: %.1f tok/s",
    latency_results["mean_latency_s"],
    latency_results["std_latency_s"],
    latency_results["throughput_tokens_per_s"],
)
print(f"\nLatência média: {latency_results['mean_latency_s']:.3f}s ± {latency_results['std_latency_s']:.3f}s")
print(f"Throughput: {latency_results['throughput_tokens_per_s']:.1f} tokens/s")

## 9. Benchmark de Consumo de Memória

In [ ]:
import psutil

def measure_memory(
    model: nn.Module,
    device: torch.device,
) -> Dict[str, float]:
    """
    Measure model memory consumption on the target device.

    Parameters
    ----------
    model : nn.Module
        Model to measure.
    device : torch.device
        Compute device.

    Returns
    -------
    dict
        Memory statistics with keys: 'gpu_allocated_gib', 'gpu_reserved_gib',
        'cpu_rss_gib', 'param_mem_gib'.
    """
    # Tamanho dos parâmetros em memória
    param_mem = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**3

    result: Dict[str, float] = {"param_mem_gib": param_mem}

    if device.type == "cuda":
        torch.cuda.synchronize()
        result["gpu_allocated_gib"] = torch.cuda.memory_allocated() / 1024**3
        result["gpu_reserved_gib"]  = torch.cuda.memory_reserved() / 1024**3
    else:
        result["gpu_allocated_gib"] = 0.0
        result["gpu_reserved_gib"]  = 0.0

    proc = psutil.Process(os.getpid())
    result["cpu_rss_gib"] = proc.memory_info().rss / 1024**3

    return result


memory_results = measure_memory(model, DEVICE)

logger.info("Consumo de memória:")
for k, v in memory_results.items():
    logger.info("  %-25s: %.4f GiB", k, v)

print("\nConsumo de Memória:")
for k, v in memory_results.items():
    print(f"  {k:<30}: {v:.4f} GiB")

## 10. Validação dos Critérios de Aceite

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader

def compute_perplexity_fast(
    model: nn.Module,
    tokenizer,
    device: torch.device,
    max_batches: int = 30,
) -> float:
    """
    Compute model perplexity on WikiText-2 validation set.

    Parameters
    ----------
    model : nn.Module
        Evaluation model.
    tokenizer : PreTrainedTokenizer
        Tokenizer compatible with the model.
    device : torch.device
        Compute device.
    max_batches : int, optional
        Maximum evaluation batches. Default 30.

    Returns
    -------
    float
        perplexity score.
    """
    val = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
    tok_val = val.map(
        lambda e: tokenizer(e["text"], truncation=True, max_length=512,
                            padding="max_length", return_tensors=None),
        batched=True, remove_columns=val.column_names,
    )
    tok_val.set_format(type="torch")
    dl = DataLoader(tok_val, batch_size=4)

    model.eval()
    total_nll, total_tok = 0.0, 0
    with torch.no_grad():
        for i, b in enumerate(dl):
            if i >= max_batches: break
            ids  = b["input_ids"].to(device)
            mask = b["attention_mask"].to(device)
            labs = ids.clone(); labs[mask == 0] = -100
            out  = model(input_ids=ids, attention_mask=mask, labels=labs)
            n    = (labs != -100).sum().item()
            total_nll += out.loss.item() * n
            total_tok += n

    return math.exp(total_nll / max(total_tok, 1))


logger.info("Calculando perplexidade do modelo exportado...")
final_ppl = compute_perplexity_fast(model, tokenizer, DEVICE)
ppl_degradation = (final_ppl - TEACHER_PPL) / max(TEACHER_PPL, 1e-8) if TEACHER_PPL < float("inf") else 0.0

logger.info("Perplexidade final: %.4f | Degradação vs. baseline: %.2f%%", final_ppl, 100 * ppl_degradation)

# Verificação de colapso de logits
sample_input = tokenizer(BENCHMARK_PROMPT, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    logits = model(**sample_input).logits
logit_std = logits.std().item()
collapsed = logit_std < ACCEPT_CRIT.get("logit_collapse_threshold", 1e-4)

# Relatório de critérios
accept_report: Dict[str, Any] = {
    "perplexity_teacher":    TEACHER_PPL,
    "perplexity_final":      final_ppl,
    "ppl_degradation_pct":   100 * ppl_degradation,
    "within_ppl_limit":      ppl_degradation <= ACCEPT_CRIT.get("max_perplexity_degradation_pct", 10.0) / 100,
    "logit_std":             logit_std,
    "logit_collapsed":       collapsed,
    "latency_mean_s":        latency_results["mean_latency_s"],
    "throughput_tps":        latency_results["throughput_tokens_per_s"],
    "param_mem_gib":         memory_results["param_mem_gib"],
    "gpu_allocated_gib":     memory_results.get("gpu_allocated_gib", 0.0),
}

print("\n" + "="*60)
print(" RELATÓRIO DE CRITÉRIOS DE ACEITE")
print("="*60)
for k, v in accept_report.items():
    print(f"  {k:<35}: {v}")
print("="*60)

if accept_report["within_ppl_limit"] and not collapsed:
    logger.info("TODOS OS CRITÉRIOS DE ACEITE ATENDIDOS. Modelo aprovado para deploy.")
else:
    logger.warning("UM OU MAIS CRITÉRIOS NÃO FORAM ATENDIDOS. Revisar antes do deploy.")

In [ ]:
# Persistir relatório final consolidado
final_report: Dict[str, Any] = {
    "project": "Hybrid Multimodal BitNet Architecture",
    "export_format": EXPORT_FORMAT,
    "quant_type": QUANT_TYPE,
    "acceptance_report": accept_report,
    "latency_benchmark": latency_results,
    "memory_benchmark": memory_results,
    "phase_history": phase_metrics,
}

report_path = METRICS_DIR / "phase4_final_report.json"
with open(report_path, "w") as f:
    json.dump(final_report, f, indent=2, default=str)

# Copiar também para o diretório de exportação
shutil.copy(report_path, EXPORT_DIR / "deployment_report.json")

logger.info("Relatório final persistido em: %s", report_path)
print(f"\nRelatório salvo em: {report_path}")

## 12. Conclusões e Sumário Executivo

### Pipeline Completo — Sumário

| Fase | Notebook | Objetivo | Status |
|------|----------|----------|--------|
| 0 | `00_baseline_teacher.ipynb` | Teacher FP16, interfaces, critérios de aceite | ✓ |
| 1 | `01_connector_pretraining.ipynb` | Pré-treinamento do conector MLP isolado | ✓ |
| 2 | `02_multimodal_alignment.ipynb` | Distilação + alignment loss + descongelamento seletivo | ✓ |
| 3 | `03_ternary_transition.ipynb` | QAT contínuo → regime ternário {-1,0,+1} | ✓ |
| 4 | `04_export_deploy.ipynb` | Exportação .safetensors, bitnet.cpp, benchmarks | ✓ |

### Configuração Final Implementada

> **Encoder especialista em FP16/BF16 + conector MLP (BF16) + núcleo BitNet ternário {-1,0,+1} + cabeças por tarefa (BF16/FP16) + QAT contínuo + distilação seletiva.**

Esta configuração maximiza a eficiência de inferência ao concentrar a compressão extrema no backbone de linguagem, preservando a qualidade de percepção nos encoders e a sensibilidade numérica nas cabeças de tarefa, conforme a Recomendação Final da especificação técnica (Seção 8).

### Comando de Inferência bitnet.cpp

Após build e conversão bem-sucedidos:

```bash
# Inferência via bitnet.cpp
python BitNet/run_inference.py \
    -m /content/drive/MyDrive/multimodal-ternary-llm/export/ternary_backbone.safetensors \
    -p "Your prompt here" \
    -n 256 \
    -temp 0.7
```